# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa 
This notebook showcases loading, exploration, and processing of a dataset using the `mlcroissant` library. The dataset contains surveys of mental health indicators from residents in Kilifi County, Kenya. It demonstrates use of Croissant schema for record sets, fields, and column access with unique `@id` referencing.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

# For visualization
!pip install matplotlib seaborn

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Display dataset name and description from metadata
print("{}: {}".format(dataset.metadata.name, dataset.metadata.description))

## 2. Data Overview
Review available record sets, their `@id`s, and fields for data exploration.

**Note:** In Croissant datasets, each entity (record set, field, column) is uniquely referenced by its `@id`.

In [ ]:
# List all record sets with their @id and fields
record_sets = dataset.metadata.recordSet
if not record_sets:
    print("No record sets found in metadata. Dataset loading with records directly.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        print("Fields:")
        for field in rs['field']:
            print(f"  Field @id: {field['@id']}, name: {field.get('name')} (type: {field.get('dataType')})")
        print('-'*30)

### Get a sample record from a specified RecordSet

If record sets exist, print a few sample records by `@id`. Otherwise, load from the dataset's default record set.

In [ ]:
# Find a record set @id (for demonstration, retrieve the first one if available)
if record_sets:
    main_record_set_id = record_sets[0]['@id']
else:
    main_record_set_id = None

# Sample records
if main_record_set_id:
    print(f"Sample records from RecordSet @id: {main_record_set_id}")
    for idx, record in enumerate(dataset.records(record_set=main_record_set_id)):
        print(record)
        if idx >= 2:
            break
else:
    print("Loading sample records from the dataset's default record set.")
    for idx, record in enumerate(dataset.records()):
        print(record)
        if idx >= 2:
            break

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

**All entities and references are by `@id`.**

If multiple record sets are present, all are loaded. Otherwise, load from the default record set.

In [ ]:
# Extract records into DataFrames
dataframes = {}

if record_sets:
    record_set_ids = [rs['@id'] for rs in record_sets]
    for rs_id in record_set_ids:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"DataFrame for RecordSet @id: {rs_id} loaded with shape {df.shape}")
else:
    # Assume single default record set
    default_rs_id = 'default_recordset'
    records = list(dataset.records())
    df = pd.DataFrame(records)
    dataframes[default_rs_id] = df
    record_set_ids = [default_rs_id]
    print(f"DataFrame for Default RecordSet loaded with shape {df.shape}")
# Display available columns
selected_recordset_id = record_set_ids[0]
print(f"Available fields/columns in DataFrame for {selected_recordset_id}:")
print(dataframes[selected_recordset_id].columns.tolist())
dataframes[selected_recordset_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records by criteria, normalizing numeric fields, and grouping by categorical attributes.

**Example: Filter GAD-7 score > 10, normalize, group by gender**

*Use field and column `@id`s for referencing throughout!*

In [ ]:
# Specify the numeric field by @id (Example: GAD-7 score)
# Assume field ids according to standardized survey: 'cr:GAD7_score', 'cr:gender'
numeric_field_id = 'cr:GAD7_score'
group_field_id = 'cr:gender'

# Use the main recordset dataframe
df = dataframes[selected_recordset_id]
# Check field existence
if numeric_field_id in df.columns:
    # Filter by GAD-7 scores above threshold
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())
    
    # Normalization
    filtered_df["normalized_{}".format(numeric_field_id)] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records (first 5):")
    print(filtered_df[[numeric_field_id, "normalized_"+numeric_field_id]].head())
    
    # Group and aggregate
    if group_field_id in filtered_df.columns:
        grouped_df = (
            filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        )
        print(f"Mean {numeric_field_id} by group {group_field_id}:")
        print(grouped_df.head())
else:
    print(f"Numeric field {numeric_field_id} not found in DataFrame columns.")

## 5. Visualization
Visualize data distributions or relationships between fields.

**Example: Distribution of GAD-7 scores and grouped means by gender.**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize GAD-7 score distribution
if numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.title('Distribution of GAD-7 scores')
    plt.xlabel('GAD-7 score (@id: {})'.format(numeric_field_id))
    plt.ylabel('Frequency')
    plt.show()

    # Boxplot by gender group
    if group_field_id in df.columns:
        plt.figure(figsize=(8, 4))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title('GAD-7 Scores by Gender')
        plt.xlabel('Gender (@id: {})'.format(group_field_id))
        plt.ylabel('GAD-7 Score')
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

This notebook demonstrated loading the Kilifi County Mental Health Survey dataset via Croissant schema using `mlcroissant`, referencing entities by their `@id`. Key features:
- **RecordSet and Fields with `@id`**: Consistent access and referencing.
- **Analysis Ready**: Data extraction, filtering, normalization, and grouping.
- **Visualization**: Distribution plots and group comparisons.

Further analysis can be performed by exploring other fields referenced by their `@id` (such as PHQ-9, PSQ, demographic fields, etc.), and extending EDA or modeling workflows accordingly.